## Basic Rotterdam-Antwerp trip  
This notebook shows the simple route without locks. There are two routes: Nieuwe Maas and Oude Maas. Some edges had to be removed or forces for these routes. The speed is fixed at 4m/s (which is correct now according to the last cell). 


## TO DO
1. X Clean
2. V Finalise simple route (add reduced depth at chosen setions) 
3. X Test with 2nd vessel
4. Copy simple route and test with Volkerak 
5. Visualise fuel consumption (kg/km and kg/h, first over volkerak then simple route) --> table 
6. Finish analysis Volkerak (extract 2 other cases)
7. Calibrate the other cases (-> create lock objects with the right parametrs) 
8. Find way to run cases in a single code (vessels, routes??, lock) 



## 0. Import libraries

In [1]:
# packages used for creating and geo-locating the graph
import networkx as nx
# New import for shallow parts 
from networkx.utils import pairwise
import shapely

# packages related to the simulation (creating the vessel, running the simulation)
import datetime
import pickle
import simpy
import opentnsim
import opentnsim.fis as fis
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core.plotutils import generate_vessel_gantt_chart
from opentnsim.graph import mixins as graph_module
from opentnsim.graph.calculations import calculate_length_of_edge

# packages related to the energy module 
from opentnsim.energy.logutils import (
    add_energy_attributes_to_eventtable,
    add_fuel_attributes_to_event_table,
    add_H2_attributes_to_event_table)

# packages needed for inspecting the output
import pandas as pd
import geopandas as gpd

# packages needed for plotting
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# plot libraries
import folium

import numpy as np

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 2.2.2.dev47+gbe28de4fe.d20260122


In [2]:
%load_ext autoreload
%autoreload 2

## 0. Load useful methods 

### A) Network functions

In [3]:
# Method to remove edges  
def safe_remove_edge(G, u, v):
    if G.has_edge(u, v):
        G.remove_edge(u, v)
        G.remove_edge(v,u)
        print(f"Removed edge: {u} -> {v} (both directions)")
    else:
        print(f"Edge {u} -> {v} does not exist (already removed).")

In [4]:
# Method to find the closest node
def find_closest_node(G,point):
    distances, idx = nodes_gdf.sindex.nearest(point)
    # return nodes_gdf.iloc[idx[0]].name
    return nodes_gdf.iloc[idx[0]]

### Other methods
Segers also introduces these methods. But I believe those are not needed in the current opentnsim version in combination with the energy module. 

B) Metods to calculate vessel properties <br>
C) Methods to calculate the resistance, power and corresponding emissions <br>
D) Calculate the energy use and emission per time step

### 1. Define object classes

--> moved this to later just before 3. Run simulation 

### 2. Create graph
Next we create a network (a graph) along which the vessel can move. For this case we use the Fairway Information System graph, and make the vessel sail from a container terminal at the Maasvlakte to the Kreekrak locks

#### 2.1 Load FIS graph

In [5]:
with open('FG_correct_distances.pickle', 'rb') as handle:
    FG = pickle.load(handle)

In [6]:
# Printing the data per edge to see what the names are. Check if there is water depth info
for u, v, data in FG.edges(data=True):
    print (u, "->", v)
    print(data)
    break

8861095 -> 8864054
{'GeoType': 'section', 'Name': 'Vaarwegvak van 0 tot 2 - H', 'Length': 2.346, 'GeneralDepth': nan, 'GeneralLength': nan, 'GeneralWidth': nan, 'SeaFairingDepth': nan, 'PushedLength': nan, 'PushedWidth': nan, 'GeneralHeight': nan, 'SeaFairingLength': nan, 'SeaFairingWidth': nan, 'CoupledLength': nan, 'CoupledWidth': nan, 'PushedDepth': nan, 'WidePushedDepth': nan, 'CoupledDepth': nan, 'WidePushedLength': nan, 'WidePushedWidth': nan, 'SeaFairingHeight': nan, 'Id_navigability': nan, 'Classification': nan, 'Code': nan, 'Description': nan, 'length_deg': nan, 'length': 0.0255813258548747, 'Wkt': 'LINESTRING (3.54535894046351 51.727661900382, 3.5602008295784 51.742791566037, 3.56379034355024 51.7453141508194)', 'StartJunctionId': '8861095', 'EndJunctionId': '8864054', 'subgraph': 0, 'length_m': 2345.572872686835, 'geometry': <LINESTRING (3.545 51.728, 3.56 51.743, 3.564 51.745)>}


In [7]:
# FG.nodes[('B4705_B')]['LINESTRING']
FG.nodes[('B4705_B')]

{'n': 'B4705_B',
 'X': 4.485519786010358,
 'Y': 51.90947209143337,
 'geometry': <POINT (4.486 51.909)>,
 'Wkt': 'POINT (4.4855197860103582 51.9094720914333720)'}

In [8]:
# Test to print specific edge data
for u, v, data in FG.edges(data=True):
    if u == 8865973 and v == 8867633:
        print(data)
    else: 
        print("Edge does not exist?")
        break

Edge does not exist?


#### 2.2 Remove edges 
Edges need to be removed to ensure that the vessels sail representative routes.

In [9]:
# Remove Spui (very shallow waterway so most vessels are not able to pass this) 
safe_remove_edge(FG, 'S22992990_A', 'S22992990_B')

# Remove recreational bergediepsluis (not for IWT vessels) 
safe_remove_edge(FG, 'L51819_A', 'L51819_B')

# Remove recreational lock volkerak (not for IWT vessels) 
safe_remove_edge(FG, 'B46500_A', 'B46500_B')

# Remove Rietbaan in nieuwe Maas route (this edge creates a loop)
# TO DO: find out why this removes Noord instead of Rietbaan?? --> difference not so relevant though
safe_remove_edge(FG, '8863048', '8866570') 

# Remove edge at start of shallow SR part (around Noordplaat Island) 
safe_remove_edge(FG, "8862801", "8862255")


Removed edge: S22992990_A -> S22992990_B (both directions)
Removed edge: L51819_A -> L51819_B (both directions)
Removed edge: B46500_A -> B46500_B (both directions)
Removed edge: 8863048 -> 8866570 (both directions)
Removed edge: 8862801 -> 8862255 (both directions)


#### 2.3 Plan route 

In [10]:
maasvlakte     = shapely.Point(4.056694, 51.947162)
volkerak       = shapely.Point(4.409331, 51.690440)
kreekrak       = shapely.Point(4.228202, 51.461103)
meerhout       = shapely.Point(5.096662, 51.098779)
antwerp        = shapely.Point(4.344243, 51.279407)
botlek         = shapely.Point(4.315013, 51.894716)
erasmus_bridge = shapely.Point(4.486020, 51.908346)
krammer        = shapely.Point(4.160114, 51.661751)
hansweert      = shapely.Point(4.010556, 51.455824)
Nieuwe_Maas1   = shapely.Point(4.333923,51.8946779)
Nieuwe_Maas2   = shapely.Point(4.619175, 51.889403)
Noord1         = shapely.Point(4.622453,51.8854942)
Noord2         = shapely.Point(4.669784,51.8240557)
# Scheldt_Rhine1 = shapely.Point(4.236250,51.6338234)
# Scheldt_Rhine1 = shapely.Point(4.217522178587546,51.62898886759566) 
Scheldt_Rhine1 = shapely.Point(4.194823569846336,51.614934177179784) 
Scheldt_Rhine2 = shapely.Point(4.220736,51.4943026)
Dort1          = shapely.Point(4.621678,51.7993701) 
Dort2          = shapely.Point(4.259127,51.6447776)
# CHANGE THESE TO SKIP BERENDRECHTSLUIS 
antwerp_ZB     = shapely.Point(4.262775,51.3438188)
antwerp_SR     = shapely.Point(4.296340,51.3441803)

In [11]:
nodes_gdf = gpd.GeoDataFrame(FG.nodes.values(), index=FG.nodes.keys())

In [12]:
# Find closest nodes for the locations
# Start locations
maasvlakte_node  = find_closest_node(FG,maasvlakte)
botlek_node      = find_closest_node(FG,botlek)
# Locks
volkerak_node   = find_closest_node(FG,volkerak)
kreekrak_node    = find_closest_node(FG,kreekrak)
krammer_node     = find_closest_node(FG,krammer)
hansweert_node   = find_closest_node(FG,hansweert)
# End locations
antwerp_node     = find_closest_node(FG,antwerp)
meerhout_node    = find_closest_node(FG,meerhout)
# Nieuwe maas route needs to pass the Erasmus bridge
erasmus_bridge_node      = find_closest_node(FG,erasmus_bridge)
# Extra locations for shallow water routes
# Nieuwe_Maas1_node = find_closest_node(FG,Nieuwe_Maas1)
# Nieuwe_Maas2_node = find_closest_node(FG,Nieuwe_Maas2)
Noord1_node = find_closest_node(FG,Noord1)
Noord2_node = find_closest_node(FG,Noord2)
Scheldt_Rhine1_node = find_closest_node(FG,Scheldt_Rhine1)
Scheldt_Rhine2_node = find_closest_node(FG,Scheldt_Rhine2)    
Dort1_node = find_closest_node(FG,Dort1)
Dort2_node = find_closest_node(FG,Dort2)
antwerp_ZB_node = find_closest_node(FG,antwerp_ZB)
antwerp_SR_node = find_closest_node(FG,antwerp_SR)

In [13]:
# Use network x to create the different routes
route_OM_SR = nx.shortest_path(FG, botlek_node.name, antwerp_SR_node.name, weight='length_m')

route_NM_SR = (
    nx.shortest_path(FG, botlek_node.name, erasmus_bridge_node.name, weight='length_m')
    + nx.shortest_path(FG, erasmus_bridge_node.name, antwerp_SR_node.name, weight='length_m')[1:])

route_OM_ZB = (
    nx.shortest_path(FG, botlek_node.name, hansweert_node.name, weight='length_m')
    + nx.shortest_path(FG, hansweert_node.name, antwerp_ZB_node.name, weight='length_m')[1:])

route_NM_ZB = (
    nx.shortest_path(FG, botlek_node.name, erasmus_bridge_node.name, weight='length_m')
    +nx.shortest_path(FG, erasmus_bridge_node.name, hansweert_node.name, weight='length_m')[1:]
    + nx.shortest_path(FG, hansweert_node.name, antwerp_ZB_node.name, weight='length_m')[1:])

shallow_NM = nx.shortest_path(FG, Noord1_node.name, Noord2_node.name, weight='length_m')
shallow_SR = nx.shortest_path(FG, Scheldt_Rhine1_node.name, Scheldt_Rhine2_node.name, weight='length_m')

Dort = nx.shortest_path(FG, Dort1_node.name, Dort2_node.name, weight='length_m')

ZB = (
    nx.shortest_path(FG, volkerak_node.name, hansweert_node.name, weight='length_m') 
    + nx.shortest_path(FG, hansweert_node.name, antwerp_ZB_node.name, weight='length_m')[1:])

SR = nx.shortest_path(FG, volkerak_node.name, antwerp_SR_node.name, weight='length_m')

### 2.2 Create water depth info on edges + visualize on network 

In [14]:
shallow_NM_set = set(shallow_NM)
shallow_SR_set = set(shallow_SR)
# shallow_SR_set.update({"8862801", "8864138"})
# shallow_SR_set.update({"8864138", "8862801"})
# shallow_SR_set.update({"8865797 ", "8862801"})


for u, v in FG.edges():
    FG.edges[u, v]["GeneralDepth"] = 6
    
    # Reduce water depth for shallow parts of the route
    # Note that this also reduced the water depth for the branches of the route
    if u in shallow_NM_set or v in shallow_NM_set:
        FG.edges[u, v]["GeneralDepth"] = 4
    if u in shallow_SR_set or v in shallow_SR_set:
        FG.edges[u, v]["GeneralDepth"] = 4


In [15]:
print(shallow_NM_set)
print(shallow_SR_set)
# node 8862801 to node 8864138 is missing in shallow SR_set

{'B43523_B', '8865140', 'B43523_A', 'Berth149', '8863131', '8866686', '8863048', '8861666', '8866570'}
{'8868095', 'B37463_B', '8865574', 'B30667_A', 'B37463_A', '8866632', 'B30667_B', 'B9108_B', 'B9108_A', '8868133', '8867083'}


#### Map: route

In [16]:
# Create a map centered between the two points
m = folium.Map(location=[51.83, 4.33], zoom_start = 10, tiles="cartodb positron")

for edge in FG.edges(data = True):
    points_x = list(edge[2]["geometry"].coords.xy[0])
    points_y = list(edge[2]["geometry"].coords.xy[1])
    
    line = []
    for i, _ in enumerate(points_x):
        line.append((points_y[i], points_x[i]))
    
    if edge[0] in route_NM_SR and edge[1] in route_NM_SR:
        folium.PolyLine(line, color = "blue", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    if edge[0] in route_OM_SR and edge[1] in route_OM_SR:
        folium.PolyLine(line, color = "orange", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    if edge[0] in ZB and edge[1] in ZB:
        folium.PolyLine(line, color = "red", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    if edge[0] in SR and edge[1] in SR:
        folium.PolyLine(line, color = "green", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    if edge[0] in Dort and edge[1] in Dort:
        folium.PolyLine(line, color = "grey", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    # if edge[0] in shallow_NM and edge[1] in shallow_NM:
    #     folium.PolyLine(line, color = "pink", weight = 3, dash_array="15,15", popup = edge[2]["StartJunctionId"]).add_to(m)
    # if edge[0] in shallow_SR and edge[1] in shallow_SR:
    #     folium.PolyLine(line, color = "pink", weight = 3, dash_array="15,15", popup = edge[2]["StartJunctionId"]).add_to(m)
    # if edge[0] in shallow_NM_set and edge[1] in shallow_NM_set:
    if edge[0] in shallow_NM_set:
        folium.PolyLine(line, color = "pink", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    # if edge[0] in shallow_SR_set and edge[1] in shallow_SR_set:
    if edge[0] in shallow_SR_set:
        folium.PolyLine(line, color = "pink", weight = 3, popup = edge[2]["EndJunctionId"]).add_to(m)

    # else:
        # folium.PolyLine(line, color = "black", weight = 1, popup = edge[2]["StartJunctionId"]).add_to(m)

# Markers
# folium.CircleMarker(location=[botlek_node.Y, botlek_node.X],radius=6,color='purple',fill=True,fill_color='purple',fill_opacity=1,popup='Botlek').add_to(m)
# folium.CircleMarker(location=[antwerp_node.Y, antwerp_node.X],radius=6,color='purple',fill=True,fill_color='purple',fill_opacity=1,popup='Antwerp').add_to(m)
folium.CircleMarker(location=[volkerak_node.Y, volkerak_node.X],radius=8,color='purple',fill=True,fill_color='purple',fill_opacity=1,popup='Volkerak').add_to(m)
folium.CircleMarker(location=[kreekrak_node.Y, kreekrak_node.X],radius=8,color='purple',fill=True,fill_color='purple',fill_opacity=1,popup='Kreekrak').add_to(m)
folium.CircleMarker(location=[krammer_node.Y, krammer_node.X],radius=8,color='purple',fill=True,fill_color='purple',fill_opacity=1,popup='Krammer').add_to(m)
folium.CircleMarker(location=[hansweert_node.Y, hansweert_node.X],radius=8,color='purple',fill=True,fill_color='purple',fill_opacity=1,popup='Hansweert').add_to(m)

# folium.CircleMarker(location=[Noord1_node.Y, Noord1_node.X],radius=8,color='brown',fill=True,fill_color='brown',fill_opacity=0.7,popup='Start shallow NM').add_to(m)
# folium.CircleMarker(location=[Noord2_node.Y, Noord2_node.X],radius=8,color='brown',fill=True,fill_color='brown',fill_opacity=0.7,popup='End shallow NM').add_to(m)
# folium.CircleMarker(location=[Scheldt_Rhine1_node.Y, Scheldt_Rhine1_node.X],radius=8,color='brown',fill=True,fill_color='brown',fill_opacity=0.7,popup='Start shallow SR').add_to(m)
# folium.CircleMarker(location=[Scheldt_Rhine2_node.Y, Scheldt_Rhine2_node.X],radius=8,color='brown',fill=True,fill_color='brown',fill_opacity=0.7,popup='End shallow SR').add_to(m)

# Display the map
m

In [17]:
# Create a map centered between the two points
m = folium.Map(location=[51.83, 4.33], zoom_start = 10, tiles="cartodb positron")

for edge in FG.edges(data = True):
    points_x = list(edge[2]["geometry"].coords.xy[0])
    points_y = list(edge[2]["geometry"].coords.xy[1])
    
    line = []
    for i, _ in enumerate(points_x):
        line.append((points_y[i], points_x[i]))
    
    # if edge[0] in route_NM_SR and edge[1] in route_NM_SR:
    #     folium.PolyLine(line, color = "blue", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    # if edge[0] in route_OM_SR and edge[1] in route_OM_SR:
    #     folium.PolyLine(line, color = "orange", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    if edge[0] in route_NM_ZB and edge[1] in route_NM_ZB:
        folium.PolyLine(line, color = "green", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    if edge[0] in route_OM_ZB and edge[1] in route_OM_ZB:
        folium.PolyLine(line, color = "red", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    if edge[0] in shallow_NM and edge[1] in shallow_NM:
        # folium.PolyLine(line, color = "darkgreen", weight = 3, dash_array="10,10", popup = edge[2]["StartJunctionId"]).add_to(m)
        folium.PolyLine(line, color = "darkgreen", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    # if edge[0] in shallow_SR and edge[1] in shallow_SR:
    #     folium.PolyLine(line, color = "brown", weight = 3, popup = edge[2]["StartJunctionId"]).add_to(m)
    # else:
        # folium.PolyLine(line, color = "black", weight = 1, popup = edge[2]["StartJunctionId"]).add_to(m)

# Markers
# folium.CircleMarker(location=[botlek_node.Y, botlek_node.X],radius=8,color='green',fill=True,fill_color='green',fill_opacity=0.7,popup='Botlek').add_to(m)
# folium.CircleMarker(location=[antwerp_node.Y, antwerp_node.X],radius=8,color='green',fill=True,fill_color='green',fill_opacity=0.7,popup='Antwerp').add_to(m)
folium.CircleMarker(location=[volkerak_node.Y, volkerak_node.X],radius=6,color='blue',fill=True,fill_color='blue',fill_opacity=1,popup='Volkerak').add_to(m)
# folium.CircleMarker(location=[kreekrak_node.Y, kreekrak_node.X],radius=8,color='blue',fill=True,fill_color='blue',fill_opacity=0.7,popup='Kreekrak').add_to(m)
folium.CircleMarker(location=[krammer_node.Y, krammer_node.X],radius=6,color='blue',fill=True,fill_color='blue',fill_opacity=1,popup='Krammer').add_to(m)
folium.CircleMarker(location=[hansweert_node.Y, hansweert_node.X],radius=6,color='blue',fill=True,fill_color='blue',fill_opacity=1,popup='Hansweert').add_to(m)

# folium.CircleMarker(location=[Noord1_node.Y, Noord1_node.X],radius=8,color='brown',fill=True,fill_color='brown',fill_opacity=0.7,popup='Start shallow NM').add_to(m)
# folium.CircleMarker(location=[Noord2_node.Y, Noord2_node.X],radius=8,color='brown',fill=True,fill_color='brown',fill_opacity=0.7,popup='End shallow NM').add_to(m)
# folium.CircleMarker(location=[Scheldt_Rhine1_node.Y, Scheldt_Rhine1_node.X],radius=8,color='brown',fill=True,fill_color='brown',fill_opacity=0.7,popup='Start shallow SR').add_to(m)
# folium.CircleMarker(location=[Scheldt_Rhine2_node.Y, Scheldt_Rhine2_node.X],radius=8,color='brown',fill=True,fill_color='brown',fill_opacity=0.7,popup='End shallow SR').add_to(m)

# Display the map
m